# **Template analisi dati — Laboratorio di Segnali e Sistemi**
`File ▸ Salva una copia in Drive`, poi esegui **Setup**, compila **Configurazione** e usa le **Ricette** che ti servono (cancella le altre). Le funzioni (`fit`, `propaga`, `media_pesata`, `grafico_fit`, ...) sono nella libreria: `help(fit)` ne mostra la guida.

In [ ]:
# ============================== SETUP ==========================================
REPO      = "git+https://github.com/luzifrancesco12-design/physics-toolbox.git"
VERSIONE  = ""        # "" = ultima versione; per bloccarne una: "@v0.1.0"
USA_LATEX = False     # True = font LaTeX reali nei grafici (+2-3 min)

!pip install -q --upgrade --force-reinstall --no-deps {REPO}{VERSIONE}
if USA_LATEX:
    !apt-get update -qq && apt-get install -y -qq texlive-latex-extra texlive-fonts-recommended dvipng cm-super > /dev/null

from physics_toolbox import *          # BasePlotter, BodePlotter, Tabelle, fit, propaga, fmt, ...
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

### Configurazione
Una voce per tabella: `foglio` = nome scheda o indice 0-based, `colonne` = (prima, ultima), `riga_intestazioni` = indice 0-based (riga 9 del foglio → `8`).

In [ ]:
URL_FOGLIO = "INCOLLA_QUI_L_URL_DEL_FOGLIO"     # il foglio deve essere "Chiunque abbia il link > Visualizzatore"

carica = Tabelle(URL_FOGLIO, {
    "misure": dict(foglio=0,            colonne=("A", "D"), riga_intestazioni=8),
    "bode":   dict(foglio="Incertezze", colonne=("A", "I"), riga_intestazioni=8),
})

### Ricetta A — Dati con barre d'errore

In [ ]:
TAB, C_X, C_Y, C_SY = "misure", "X", "Y", "Err Y"

df = carica(TAB)
x, y, sy = pulisci(col(df, C_X), col(df, C_Y), col(df, C_SY))

g = BasePlotter(figsize=(9, 5))
g.add_grid(0, xscale="linear", yscale="linear")          # "log" per scala logaritmica
g.set_labels(r"$x$ [unità]", r"$y$ [unità]")
g.plot_error(x, y, yerr=sy, label="Dati", linestyle="none")
g.add_legend()
salva(g, "dati.png"); g.show()

### Ricetta B — Media pesata su un intervallo

In [ ]:
TAB, C_X, C_Y, C_SY = "misure", "X", "Y", "σ Y"
INTERVALLO = (0, 10)

df = carica(TAB)
x, y, sy = pulisci(col(df, C_X), col(df, C_Y), col(df, C_SY))
m = tra(x, *INTERVALLO)
media, s_media, chi2r = media_pesata(y[m], sy[m])
print(f"Media pesata = {fmt(media, s_media)}   ({m.sum()} punti, chi2 ridotto = {chi2r:.2f})")

### Ricetta C — Fit di un modello qualsiasi (con residui)

In [ ]:
TAB, C_X, C_Y, C_SY = "misure", "X", "Y", "σ Y"
INTERVALLO = None            # es. (0, 50) per fittare solo una parte

def modello(x, a, b):        # <-- il tuo modello, es. a*np.exp(-x/b)
    return a * x + b
P0, NOMI = None, ["a", "b"]

df = carica(TAB)
x, y, sy = pulisci(col(df, C_X), col(df, C_Y), col(df, C_SY))
if INTERVALLO:
    m = tra(x, *INTERVALLO); x, y, sy = x[m], y[m], sy[m]

res = fit(modello, x, y, sy, p0=P0, nomi=NOMI)
g = grafico_fit(x, y, sy, res, xlabel=r"$x$ [unità]", ylabel=r"$y$ [unità]")
salva(g, "fit.png"); g.show()

### Ricetta D — Propagazione degli errori

In [ ]:
# Esempio: P = V^2 / R  con V = 5.00 ± 0.05 V, R = 100 ± 1 ohm
P, sP = propaga(lambda p: p[0] ** 2 / p[1], [5.00, 100.0], err=[0.05, 1.0])
print(f"P = {fmt(P, sP, 'W')}")
# Con i parametri di un fit (covarianza inclusa): propaga(lambda p: p[0]/p[1], res["popt"], cov=res["pcov"])

### Ricetta E — Bode
Blocco di `Crea Blocco Incertezze` (intestazioni riga 9; blocchi ogni 10 colonne: A–I, K–S, U–AC). Nei vecchi fogli l'errore può chiamarsi `Err Gain H (dB)`.

In [ ]:
TAB = "bode"
C_FREQ, C_GAIN, C_ERR = "Frequenza (Hz)", "Gain H (dB)", "σ Gain H (dB)"

df = carica(TAB)
f, gdb, sg = pulisci(col(df, C_FREQ), col(df, C_GAIN), col(df, C_ERR))

bode = BodePlotter(figsize=(10, 5), show_phase=False)
bode.plot_bode(f, gdb, magnitude_err=sg, is_db_err=True, capsize=0, label_mag="Dati sperimentali")
salva(bode, "bode.png"); bode.show()

### Ricetta F — Frequenze di taglio (plateau − 3 dB) da fit locali
Plateau (media pesata) → soglia → fit lineare in $\log_{10} f$ su ogni fronte → `x_da_y`.

In [ ]:
TAB = "bode"
C_FREQ, C_GAIN, C_ERR = "Frequenza (Hz)", "Gain H (dB)", "σ Gain H (dB)"
PLATEAU, TAGLIO_DB = (1e3, 1e4), 3.0
TAGLI = [{"nome": "f_L", "range": (50, 500)},          # un elemento per ogni taglio
         {"nome": "f_H", "range": (5e4, 5e5)}]

def modello_log(x, m, c):
    return m * np.log10(x) + c

df = carica(TAB)
f, gdb, sg = pulisci(col(df, C_FREQ), col(df, C_GAIN), col(df, C_ERR))

plat, s_plat, chi2r = media_pesata(gdb[tra(f, *PLATEAU)], sg[tra(f, *PLATEAU)])
soglia = plat - TAGLIO_DB
print(f"Plateau = {fmt(plat, s_plat, 'dB')} (chi2 ridotto {chi2r:.2f});  soglia = {soglia:.3f} dB")

tagli = {}
for i, t in enumerate(TAGLI, 1):
    lo, hi = t["range"]; m = tra(f, lo, hi)
    print(f"{t['nome']}:")
    res = fit(modello_log, f[m], gdb[m], sg[m], nomi=["m", "c"])
    tagli[t["nome"]] = x_da_y(res, soglia, s_plat, lo, hi)
    print(f"  --> {t['nome']} = {fmt(*tagli[t['nome']], 'Hz')}")
    g = grafico_fit(f[m], gdb[m], sg[m], res, xlabel=r"$f\ \mathrm{[Hz]}$", ylabel=r"$|H|\ \mathrm{[dB]}$",
                    xscale="log", residui=False)
    g.add_hline(soglia, color="green", label="Soglia"); g.add_vline(tagli[t["nome"]][0], color="red", label=f"${t['nome']}$")
    g.add_legend(); salva(g, f"fit_taglio_{i}.png"); g.show()

b = BodePlotter(figsize=(10, 6))
b.plot_bode(f, gdb, magnitude_err=sg, is_db_err=True, capsize=0, label_mag="Dati sperimentali")
b.add_hline(plat, color="firebrick", label="Plateau");  b.ax_mag.axhspan(plat - s_plat, plat + s_plat, color="firebrick", alpha=0.15)
b.add_hline(soglia, color="forestgreen", label=rf"Soglia $-{TAGLIO_DB:g}\,\mathrm{{dB}}$"); b.ax_mag.axhspan(soglia - s_plat, soglia + s_plat, color="forestgreen", alpha=0.15)
for nome, (f0, sf0) in tagli.items():
    b.add_vline(f0, color="darkgreen", linestyle="solid", label=f"${nome} = $" + fmt(f0, sf0, "Hz", latex=True))
b.ax_mag.set_xlim(f.min() / 1.5, f.max() * 1.5); b.add_legend()
salva(b, "bode_final.png"); b.show()

### Download dei grafici (zip)

In [ ]:
scarica_figure("grafici.zip")